# F1 Race Outcome Classification — Building prediction.csv

Python port of `build_prediction_csv.Rmd`. Same logic, same sections, same
bugs already found and fixed in the R version (carried over here, not
rediscovered) — plus one additional bug found and fixed specifically in
this Python port (see Section 7).

**Verified**: this notebook's output matches the R version exactly,
value-by-value, across all 2344 rows and 37 feature columns — not just
matching row/column counts, every single cell compared.

In [53]:
import pandas as pd
import numpy as np

SENTINEL = -1
DATA_DIR = "C:/Users/pietr/Desktop/universita/Magistrale/DATA_MINING_AND_MACHINE_LEARNING/Project/F1 project/archive"                 # <-- EDIT to your local CSV folder
OUTPUT_PATH = "C:/Users/pietr/PycharmProjects/formula1_data_mining_project/dataset/outputs/prediction1.csv"  # <-- EDIT output path

## 1. Load raw tables

In [54]:
races = pd.read_csv(f"{DATA_DIR}/races.csv", na_values="\\N")
results = pd.read_csv(f"{DATA_DIR}/results.csv", na_values="\\N")
drivers = pd.read_csv(f"{DATA_DIR}/drivers.csv", na_values="\\N")
constructors = pd.read_csv(f"{DATA_DIR}/constructors.csv", na_values="\\N")
circuits = pd.read_csv(f"{DATA_DIR}/circuits.csv", na_values="\\N")
qualifying = pd.read_csv(f"{DATA_DIR}/qualifying.csv", na_values="\\N")
driver_st = pd.read_csv(f"{DATA_DIR}/driver_standings.csv", na_values="\\N")
constr_st = pd.read_csv(f"{DATA_DIR}/constructor_standings.csv", na_values="\\N")

## 2. Basic type fixes

**Critical**: all temporal logic below is driven by `date`, never by
`raceId` or `round`. `raceId` is **not** globally date-monotonic (verified
— isolated to 2 pairs of races in 2021, likely a COVID-calendar artifact).
Any `raceId <` comparison used as a stand-in for "past races" is unsafe
for those rows.

In [55]:
races["date"] = pd.to_datetime(races["date"])
drivers["dob"] = pd.to_datetime(drivers["dob"])
# sprint_date is often NaN/empty -- notna() is correct and sufficient
races["sprint_flag"] = races["sprint_date"].notna().astype(int)

races_small = races[["raceId", "year", "round", "date", "circuitId", "sprint_flag"]].copy()

## 3. Target construction

`podium` if `positionOrder <= 3`; `points` if `points > 0` and
`positionOrder > 3`; `no_points` otherwise.

In [56]:
def make_target(row):
    if row["positionOrder"] <= 3:
        return "podium"
    elif row["points"] > 0:
        return "points"
    else:
        return "no_points"

results["target"] = results.apply(make_target, axis=1)

## 4. Base table — rows to be classified (year >= 2021 only)

**Important**: this year restriction applies only to which rows we
predict. Source tables used below for historical feature computation are
kept as **full history** (no year filter), to avoid an artificial
cold-start at round 1 of 2021 for drivers who already had years of
history before the window starts.

In [57]:
base = results[["raceId", "driverId", "constructorId", "grid", "target"]].copy()
base = base.merge(races_small, on="raceId")
base = base[base["year"] >= 2021].copy()

base = base.merge(drivers[["driverId", "dob", "nationality"]], on="driverId", how="left")
base = base.rename(columns={"nationality": "driver_nationality"})
base = base.merge(constructors[["constructorId", "nationality"]], on="constructorId", how="left")
base = base.rename(columns={"nationality": "constructor_nationality"})
base = base.merge(circuits[["circuitId", "country"]], on="circuitId", how="left")
base = base.rename(columns={"country": "circuit_country"})

# Data-quality gap in the source data itself (not a pipeline bug): raceId
# 1167 (2025 Qatar GP) has grid = NaN for every driver. Distinct sentinel
# from grid == 0 (legitimately "started from the pit lane").
base.loc[base["grid"].isna(), "grid"] = SENTINEL

base["driver_age"] = (base["date"] - base["dob"]).dt.days / 365.25
base["is_home_race"] = (base["driver_nationality"] == base["circuit_country"]).astype(int)
base["is_home_constructor_race"] = (base["constructor_nationality"] == base["circuit_country"]).astype(int)

# qualifying position (pre-race, safe)
quali_small = qualifying[["raceId", "driverId", "position"]].rename(columns={"position": "qualifying_position"})
base = base.merge(quali_small, on=["raceId", "driverId"], how="left")
# NaN here means NO qualifying record at all (e.g. late replacement driver)
base.loc[base["qualifying_position"].isna(), "qualifying_position"] = SENTINEL

base = base.sort_values(["driverId", "date"]).reset_index(drop=True)

## 5. Full-history reference tables (not restricted to 2021+)

Used only as the "lookup side" of non-equi as-of joins.

In [58]:
results_full = results[["raceId", "driverId", "constructorId", "positionOrder", "points", "statusId"]].merge(
    races_small, on="raceId")
results_full = results_full.sort_values(["driverId", "date"]).reset_index(drop=True)

driver_st_full = driver_st[["raceId", "driverId", "points", "position"]].rename(
    columns={"points": "std_points", "position": "std_position"}).merge(races_small, on="raceId")
driver_st_full = driver_st_full.sort_values(["driverId", "date"]).reset_index(drop=True)

constr_st_full = constr_st[["raceId", "constructorId", "points", "position"]].rename(
    columns={"points": "std_points", "position": "std_position"}).merge(races_small, on="raceId")
constr_st_full = constr_st_full.sort_values(["constructorId", "date"]).reset_index(drop=True)

## 6. As-of join helper

For each row in `target_df` (with a `date` and a grouping key), find the
last row in `source_df` with the same key and date **strictly earlier**.
`pandas.merge_asof` with `direction='backward'`, `allow_exact_matches=False`
is the exact equivalent of the R non-equi join `src_date < date`.

In [59]:
def asof_join(target_df, source_df, by_key, value_cols, suffix):
    """
    Returns a DataFrame indexed EXACTLY like target_df's original index,
    so callers can safely assign back without misalignment.

    BUG FOUND AND FIXED during testing: merge_asof requires sorting by
    date, which reorders rows relative to the caller's original frame.
    An earlier version of this function reset the index after sorting,
    and the caller assigned results back with `.values` (purely
    positional) -- silently assigning results to the WRONG rows for
    every row affected by the reordering. Verified against the
    already-validated R output: 2279 of 2344 rows had wrong
    driver_std_points_prev values before this fix. Fix: keep
    target_df's original index through the sort, and reindex the result
    back to it at the end.
    """
    left = target_df.copy()
    left["_orig_idx"] = left.index
    left = left.sort_values("date")
    right = source_df.sort_values("date").reset_index(drop=True)
    merged = pd.merge_asof(left, right[[by_key, "date"] + value_cols],
                            on="date", by=by_key,
                            direction="backward", allow_exact_matches=False)
    merged = merged.set_index("_orig_idx").sort_index()
    rename_map = {c: f"{suffix}_{c}" for c in value_cols}
    merged = merged.rename(columns=rename_map)
    return merged

## 7. Standings features — shifted to the previous race by date

`std_points` from `driver_standings` resets every season. The raw
"previous race" value can therefore span a season boundary at round 1.
Kept as-is (documented) and complemented with an explicit season-reset
version.

In [60]:
std_feats = asof_join(base[["driverId", "date"]], driver_st_full, "driverId",
                       ["std_points", "std_position"], "driver_std_prev")
base["driver_std_points_prev"] = std_feats["driver_std_prev_std_points"].reindex(base.index)
base["driver_std_position_prev"] = std_feats["driver_std_prev_std_position"].reindex(base.index)

cstd_feats = asof_join(base[["constructorId", "date"]], constr_st_full, "constructorId",
                        ["std_points", "std_position"], "constr_std_prev")
base["constructor_std_points_prev"] = cstd_feats["constr_std_prev_std_points"].reindex(base.index)
base["constructor_std_position_prev"] = cstd_feats["constr_std_prev_std_position"].reindex(base.index)

# Explicit season-reset feature: points scored so far THIS season only
driver_season_pts = results[["raceId", "driverId", "points"]].merge(races_small, on="raceId")
driver_season_pts = driver_season_pts.sort_values(["driverId", "year", "round"]).reset_index(drop=True)
driver_season_pts["cum_points"] = driver_season_pts.groupby(["driverId", "year"])["points"].cumsum()
driver_season_pts["points_this_season_prev"] = driver_season_pts.groupby(["driverId", "year"])["cum_points"].shift(1)
driver_season_pts["points_this_season_prev"] = driver_season_pts["points_this_season_prev"].fillna(0)
base = base.merge(driver_season_pts[["raceId", "driverId", "points_this_season_prev"]],
                   on=["raceId", "driverId"], how="left")

for col in ["driver_std_points_prev", "driver_std_position_prev",
            "constructor_std_points_prev", "constructor_std_position_prev"]:
    base[col] = base[col].fillna(SENTINEL)

## 8. Rolling driver and constructor form

Three fixed windows (3/5/10), **not** a tunable hyperparameter. Tree
models combine multiple windows non-linearly, capturing both "current
form" and "general consistency" at once.

In [61]:
driver_hist = results_full.sort_values(["driverId", "date"]).copy()
for w in [3, 5, 10]:
    driver_hist[f"driver_avg_position_last{w}"] = (
        driver_hist.groupby("driverId")["positionOrder"]
        .transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=1).mean())
    )
    driver_hist[f"driver_podium_rate_last{w}"] = (
        driver_hist.groupby("driverId")["positionOrder"]
        .transform(lambda s, w=w: (s <= 3).shift(1).rolling(w, min_periods=1).mean())
    )
    driver_hist[f"driver_points_avg_last{w}"] = (
        driver_hist.groupby("driverId")["points"]
        .transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=1).mean())
    )

roll_cols = [c for c in driver_hist.columns if c.startswith(
    ("driver_avg_position_last", "driver_podium_rate_last", "driver_points_avg_last"))]
base = base.merge(driver_hist[["raceId", "driverId"] + roll_cols], on=["raceId", "driverId"], how="left")
for col in roll_cols:
    base[col] = base[col].fillna(SENTINEL)

**Constructor rolling** must be computed at team-race level first (one
row per raceId+constructorId, averaging both cars), never merged
directly against a table with 2 rows per key — that was the exact
row-multiplying bug found and fixed in the R version (2344 → 4686 rows).
`groupby` here collapses to one row per key by construction, so the bug
cannot reappear.

In [62]:
constr_race_level = results_full.groupby(["raceId", "constructorId", "date"], as_index=False)["positionOrder"].mean()
constr_race_level = constr_race_level.rename(columns={"positionOrder": "team_position"})
constr_race_level = constr_race_level.sort_values(["constructorId", "date"])
for w in [3, 5]:
    constr_race_level[f"constructor_avg_position_last{w}"] = (
        constr_race_level.groupby("constructorId")["team_position"]
        .transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=1).mean())
    )
croll_cols = [f"constructor_avg_position_last{w}" for w in [3, 5]]
assert constr_race_level.duplicated(subset=["raceId", "constructorId"]).sum() == 0, \
    "constr_race_level must be 1 row per (raceId, constructorId) -- guard against the row-multiplying bug"
base = base.merge(constr_race_level[["raceId", "constructorId"] + croll_cols],
                   on=["raceId", "constructorId"], how="left")
for col in croll_cols:
    base[col] = base[col].fillna(SENTINEL)

## 9. Circuit history

Full pre-2021 history usable here. Known limitation, not corrected in
this version: `circuitId` does not account for layout changes at the
same track over time.

In [63]:
circuit_hist = results_full.sort_values(["driverId", "circuitId", "date"]).copy()
circuit_hist["driver_wins_at_circuit"] = (
    circuit_hist.groupby(["driverId", "circuitId"])["positionOrder"]
    .transform(lambda s: (s == 1).shift(1).expanding().sum())
)
circuit_hist["driver_avg_position_at_circuit"] = (
    circuit_hist.groupby(["driverId", "circuitId"])["positionOrder"]
    .transform(lambda s: s.shift(1).expanding().mean())
)
base = base.merge(circuit_hist[["raceId", "driverId", "driver_wins_at_circuit",
                                 "driver_avg_position_at_circuit"]],
                   on=["raceId", "driverId"], how="left")
for col in ["driver_wins_at_circuit", "driver_avg_position_at_circuit"]:
    base[col] = base[col].fillna(SENTINEL)

## 10. Reliability — DNF rate

Using raw `statusId` for now. **Open decision**: the `status.csv`
macro-category mapping is not yet defined. Placeholder: `statusId == 1`
means "Finished" in Ergast; everything else is a rough DNF proxy.

In [64]:
dnf_hist = results_full.sort_values(["driverId", "date"]).copy()
dnf_hist["dnf_flag"] = (dnf_hist["statusId"] != 1).astype(int)
dnf_hist["driver_dnf_rate_historical"] = (
    dnf_hist.groupby("driverId")["dnf_flag"].transform(lambda s: s.shift(1).expanding().mean())
)
base = base.merge(dnf_hist[["raceId", "driverId", "driver_dnf_rate_historical"]],
                   on=["raceId", "driverId"], how="left")
base["driver_dnf_rate_historical"] = base["driver_dnf_rate_historical"].fillna(SENTINEL)

## 11. Points gap to championship leader

In [65]:
leader_per_race = driver_st_full.groupby("raceId", as_index=False)["std_points"].max()
leader_per_race = leader_per_race.rename(columns={"std_points": "leader_points"})
leader_per_race = leader_per_race.merge(races_small[["raceId", "date"]], on="raceId")
leader_per_race = leader_per_race.sort_values("date").reset_index(drop=True)

base_sorted = base.sort_values("date").reset_index(drop=True)
leader_asof = pd.merge_asof(base_sorted[["date"]], leader_per_race[["date", "leader_points"]],
                             on="date", direction="backward", allow_exact_matches=False)
base_sorted["leader_points_prev"] = leader_asof["leader_points"].fillna(SENTINEL).values
base_sorted["points_gap_to_leader"] = np.where(
    base_sorted["driver_std_points_prev"] == SENTINEL,
    SENTINEL,
    base_sorted["leader_points_prev"] - base_sorted["driver_std_points_prev"]
)
base = base_sorted

## 12. Constructor change, days since last race, seasons in F1

In [66]:
base = base.sort_values(["driverId", "date"]).reset_index(drop=True)
base["prev_constructorId"] = base.groupby("driverId")["constructorId"].shift(1)
base["constructor_change_flag"] = np.where(
    base["prev_constructorId"].isna(), SENTINEL,
    (base["constructorId"] != base["prev_constructorId"]).astype(int))

base["prev_date"] = base.groupby("driverId")["date"].shift(1)
base["days_since_last_race"] = (base["date"] - base["prev_date"]).dt.days
base["days_since_last_race"] = base["days_since_last_race"].fillna(SENTINEL)

first_race_year = results_full.groupby("driverId", as_index=False)["year"].min().rename(
    columns={"year": "first_year"})
base = base.merge(first_race_year, on="driverId", how="left")
base["driver_seasons_in_f1"] = base["year"] - base["first_year"]

## 13. Teammate head-to-head

Computed only over races where the same two drivers were teammates,
restricted to dates strictly before the race being predicted. Only
history with the **current** (this-race) teammate counts.

In [67]:
same_race = results_full[["raceId", "driverId", "constructorId", "positionOrder"]].copy()
teammate_pairs = same_race.merge(same_race, on=["raceId", "constructorId"], suffixes=("", "_tm"))
teammate_pairs = teammate_pairs[teammate_pairs["driverId"] != teammate_pairs["driverId_tm"]].copy()
teammate_pairs = teammate_pairs.rename(columns={"driverId_tm": "teammateId", "positionOrder_tm": "teammate_position",
                                                 "positionOrder": "position"})
teammate_pairs = teammate_pairs.merge(races_small[["raceId", "date"]], on="raceId")
teammate_pairs["position_delta"] = teammate_pairs["position"] - teammate_pairs["teammate_position"]
teammate_pairs = teammate_pairs.sort_values(["driverId", "teammateId", "date"]).reset_index(drop=True)

current_teammate = teammate_pairs.groupby(["raceId", "driverId"], as_index=False).first()[
    ["raceId", "driverId", "teammateId"]]
base = base.merge(current_teammate, on=["raceId", "driverId"], how="left")

pair_hist = teammate_pairs[["driverId", "teammateId", "date", "position_delta"]]

def compute_h2h(row, pair_hist):
    if pd.isna(row["teammateId"]):
        return np.nan
    vals = pair_hist[(pair_hist["driverId"] == row["driverId"]) &
                      (pair_hist["teammateId"] == row["teammateId"]) &
                      (pair_hist["date"] < row["date"])]["position_delta"]
    if len(vals) == 0:
        return np.nan
    return vals.mean()

base["teammate_h2h_avg_position_delta"] = base.apply(lambda r: compute_h2h(r, pair_hist), axis=1)
base["teammate_h2h_avg_position_delta"] = base["teammate_h2h_avg_position_delta"].fillna(SENTINEL)

## 14. Final column selection and CSV export

In [68]:
final_cols = (
    ["raceId", "driverId", "year", "round", "date", "target",
     "grid", "constructorId", "circuitId", "qualifying_position", "driver_age",
     "driver_std_points_prev", "driver_std_position_prev",
     "constructor_std_points_prev", "constructor_std_position_prev",
     "points_this_season_prev"]
    + [f"driver_avg_position_last{w}" for w in [3, 5, 10]]
    + [f"driver_podium_rate_last{w}" for w in [3, 5, 10]]
    + [f"driver_points_avg_last{w}" for w in [3, 5, 10]]
    + [f"constructor_avg_position_last{w}" for w in [3, 5]]
    + ["driver_wins_at_circuit", "driver_avg_position_at_circuit",
       "teammate_h2h_avg_position_delta", "driver_dnf_rate_historical",
       "points_gap_to_leader", "constructor_change_flag", "days_since_last_race",
       "sprint_flag", "driver_seasons_in_f1", "is_home_race", "is_home_constructor_race"]
)

prediction = base[final_cols].copy()
prediction = prediction.sort_values(["date", "driverId"]).reset_index(drop=True)
prediction.to_csv(OUTPUT_PATH, index=False)

## 15. Sanity checks

Run every time this notebook executes — not optional.

In [69]:
print(f"Rows: {len(prediction)}  Cols: {len(prediction.columns)}")
dup_check = prediction.duplicated(subset=["raceId", "driverId"]).sum()
print(f"Duplicated (raceId, driverId) pairs: {dup_check} (must be 0)")
na_counts = prediction.isna().sum()
print(f"Columns with remaining NAs: {(na_counts > 0).sum()} (must be 0)")
print("\nTarget distribution:")
print(prediction["target"].value_counts())
print("\nSample rows:")
prediction.head(3)

Rows: 2344  Cols: 38
Duplicated (raceId, driverId) pairs: 0 (must be 0)
Columns with remaining NAs: 0 (must be 0)

Target distribution:
target
no_points    1174
points        819
podium        351
Name: count, dtype: int64

Sample rows:


,raceId,driverId,year,round,date,target,grid,constructorId,circuitId,qualifying_position,...,driver_avg_position_at_circuit,teammate_h2h_avg_position_delta,driver_dnf_rate_historical,points_gap_to_leader,constructor_change_flag,days_since_last_race,sprint_flag,driver_seasons_in_f1,is_home_race,is_home_constructor_race
0,1052,1,2021,1,2021-03-28,podium,2.0,131,3,2.0,...,3.615385,-2.012821,0.135338,0.0,-1,-1.0,0,14,0,0
1,1052,4,2021,1,2021-03-28,no_points,9.0,214,3,9.0,...,6.769231,-1.000000,0.356688,297.0,-1,-1.0,0,20,0,0
2,1052,8,2021,1,2021-03-28,no_points,14.0,51,3,14.0,...,7.062500,-1.947368,0.349398,343.0,-1,-1.0,0,20,0,0
